# Optuna inference experiment

Install the checkout with `pip install -e ".[notebook]"`. This notebook edits settings, launches the CLI, and explores saved artifacts. The reference remains in `optuna_inference_optimization_lab.ipynb`.

The example starts in simulation with W&B disabled. Real GPU runs require pinned local snapshots and verified preflight fields. Run, validate, and report share one frozen config and run directory. Repeating run resumes to the configured trial budget.


In [ ]:
import json
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/llm_optimizer").is_dir())
CONFIG = ROOT / "experiment_results" / "cli-demo-config.json"
RUN_DIR = ROOT / "experiment_results" / "cli-demo"
config = json.loads((ROOT / "docs/optuna_experiment.json").read_text())
config.update(phases=["A"], frameworks=["vllm"], concurrencies=[1],
              sampler_names=["random"], pareto_samplers=["random", "nsga2"],
              objectives={"1": ["e2e", "pareto_latency"], "128": ["e2e"]},
              run_real_gpu=False, wandb_enabled=False, validation_repeats=2)
config["demo_trials"]["A"]["1"] = 4
CONFIG.parent.mkdir(parents=True, exist_ok=True)
CONFIG.write_text(json.dumps(config, indent=2) + "\n")
config


Edit the config above before creating a run. To resume an existing run, set `CONFIG = RUN_DIR / "config.json"`. Choose a new run directory when changing controls.


In [ ]:
!llm-optimizer experiment run --config "{CONFIG}" --run-dir "{RUN_DIR}"


In [ ]:
!llm-optimizer experiment validate --config "{CONFIG}" --run-dir "{RUN_DIR}"


In [ ]:
!llm-optimizer experiment report --config "{CONFIG}" --run-dir "{RUN_DIR}"


In [ ]:
import pandas as pd

results = json.loads((RUN_DIR / "results.json").read_text())
environment = json.loads((RUN_DIR / "environment.json").read_text())
study_table = pd.DataFrame([study["summary"] for study in results["studies"]])
display(study_table)
display(pd.DataFrame(results["validation"]))
results["recommendations"]


In [ ]:
import matplotlib.pyplot as plt

for study in results["studies"]:
    complete = [t for t in study["trials"] if t["state"] == "COMPLETE"]
    if not complete:
        continue
    metrics = study["contract"]["metrics"]
    fig, ax = plt.subplots()
    if len(metrics) == 2:
        ax.scatter([t["values"][0] for t in complete], [t["values"][1] for t in complete])
        ax.set(xlabel=metrics[0], ylabel=metrics[1])
    else:
        ax.plot([t["number"] for t in complete], [t["values"][0] for t in complete], marker="o")
        ax.set(xlabel="Trial", ylabel=metrics[0])
    ax.set_title(study["name"])
    plt.show()
